In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn import datasets
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
X_val = pd.read_parquet("../data/processed/X_val.parquet")

y_train = pd.read_parquet("../data/processed/y_train.parquet")["price"]
y_val = pd.read_parquet("../data/processed/y_val.parquet")["price"]
y_test = pd.read_parquet("../data/processed/y_test.parquet")["price"]



In [2]:

X_train.head()

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,...,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,...,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,...,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,...,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,...,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True


In [3]:
X_test.shape

(266276, 21)

In [4]:
X_val.shape

(266276, 21)

In [5]:
categorical_cols = ['city',
 'engine_type',
 'frame_damaged',
 'fuel_type',
 'has_accidents',
 'make_name',
 'model_name',
 'salvage',
 'transmission',
 'trim_name',
 'wheel_system']

categoric_transformer = ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_cols)

numerical_cols = ['horsepower', 'mileage', 'owner_count', 'year']
numeric_transformer = ('num', StandardScaler(), numerical_cols)



boolean_cols =['is_new',
 'mileage_missing',
 'horsepower_missing',
 'Condition_reported',
 'new_mileage_conflict',
 'used_owner_count_missing'] 
boolean_transformer = ("bool", "passthrough", boolean_cols)

preprocessor = ColumnTransformer(
    transformers=[
        numeric_transformer,
        categoric_transformer,
        boolean_transformer
    ]
)

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


In [7]:
X_train_processed.shape


(2130205, 14147)

In [8]:
X_test_processed.shape

(266276, 14147)

In [9]:
X_val_processed.shape

(266276, 14147)

In [13]:
lr_model = LinearRegression()
lr_model.fit(X_train_processed, y_train)
y_pred = lr_model.predict(X_val_processed)
y_prediction = lr_model.predict(X_train_processed)


In [14]:
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
train_rmse = np.sqrt(mean_squared_error(y_train,y_prediction))
r2 = r2_score(y_val, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("train_rmse", train_rmse )

MAE: 3078.375036191397
RMSE: 7635.9709004696015
R²: 0.8486924235048505
train_rmse 7671.01676499368


In [17]:
baseline_price = y_train.mean()
baseline_val_pred=np.full(len(y_val), baseline_price)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_val_pred))

print("baseline_price", baseline_price)
print("baseline_val_pred", baseline_val_pred)
print("baseline_rmse", baseline_rmse)

baseline_price 29735.665494197976
baseline_val_pred [29735.6654942 29735.6654942 29735.6654942 ... 29735.6654942 29735.6654942
 29735.6654942]
baseline_rmse 19630.61871055733


In [18]:
error = y_val - y_pred
absolute_error = abs(error)

In [19]:
abs(error)


511069     1984.193235
171176     6773.955077
1773867    4079.853248
1682907    3538.910547
647827     4528.374259
              ...     
495748     7467.952715
183464     2094.174318
1152766    1009.994937
1181365    3087.701340
1259026    1077.172218
Name: price, Length: 266276, dtype: float64

In [20]:
abs(error).min()

np.float64(0.0011064693826483563)

In [21]:
abs(error).max()

np.float64(1052997.1849547136)

In [22]:
abs(error).mean()

np.float64(3078.375036191397)

In [23]:
abs(error).median()

np.float64(2114.265414203932)

In [24]:
descending_abs = abs(error).sort_values(ascending = False).head(10)

In [25]:
descending_abs.describe()

count    1.000000e+01
mean     8.033837e+05
std      2.117106e+05
min      5.273606e+05
25%      6.368978e+05
50%      7.365605e+05
75%      1.034631e+06
max      1.052997e+06
Name: price, dtype: float64

In [26]:
abs(error).sort_values(ascending = False).head(10)

2544096    1.052997e+06
2544074    1.051562e+06
2544045    1.050747e+06
1478429    9.862819e+05
488893     7.811130e+05
2610762    6.920079e+05
2256267    6.911645e+05
59100      6.188089e+05
941607     5.817939e+05
1166533    5.273606e+05
Name: price, dtype: float64

In [27]:
X_val.loc[2544096 ]

city                         Foothill Ranch
engine_type                              I4
frame_damaged                  Not Reported
fuel_type                          Gasoline
has_accidents                  Not Reported
horsepower                            180.0
is_new                                 True
make_name                     Mercedes-Benz
mileage                                 0.0
model_name                          E-Class
owner_count                             0.0
salvage                        Not Reported
transmission                              A
trim_name                   E 350 Sedan RWD
wheel_system                        Unknown
year                                   2021
mileage_missing                           0
horsepower_missing                        1
new_mileage_conflict                  False
used_owner_count_missing              False
Condition_reported                    False
Name: 2544096, dtype: object

In [28]:
predictions = pd.Series(y_pred, index=y_val.index)

In [29]:
predictions.loc[2544096]

np.float64(63713.81504528654)

In [30]:
y_val.loc[2544096]

np.float64(1116711.0)

In [31]:
error.loc[2544096]

np.float64(1052997.1849547136)

In [32]:
worst_indices = descending_abs.index

In [33]:
worst_cases = X_val.loc[
    worst_indices,
    ["year", "make_name", "model_name", "trim_name", "mileage", "horsepower", "is_new"]
].copy()

In [34]:
worst_cases["actual_price"] = y_val.loc[worst_indices]
worst_cases["predicted_price"] = predictions.loc[worst_indices]
worst_cases["absolute_error"] = descending_abs

In [35]:
worst_cases

,year,make_name,model_name,trim_name,mileage,horsepower,is_new,actual_price,predicted_price,absolute_error
2544096,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1116711.0,63713.815045,1.052997e+06
2544074,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1115276.0,63713.815045,1.051562e+06
2544045,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1114461.0,63713.815045,1.050747e+06
1478429,2015,Porsche,Rare,Unknown,1034.0,395.0,False,1225800.0,239518.107893,9.862819e+05
488893,2005,Porsche,Rare,2 Dr STD Convertible,338.0,605.0,False,984900.0,203786.967589,7.811130e+05
2610762,2006,Land Rover,LR3,V6,120000.0,216.0,False,699900.0,7892.077698,6.920079e+05
2256267,2017,Ford,GT,RWD,80.0,647.0,False,975000.0,283835.479851,6.911645e+05
59100,2005,Porsche,Rare,2 Dr STD Convertible,2300.0,605.0,False,819000.0,200191.110366,6.188089e+05
941607,1989,BMW,7 Series,750iL RWD,41308.0,300.0,False,4500.0,586293.889209,5.817939e+05
1166533,2020,Ford,F-250 Super Duty,Unknown,3.0,445.0,True,589250.0,61889.380443,5.273606e+05


In [36]:
worst_cases.sort_values("actual_price", ascending = False)

,year,make_name,model_name,trim_name,mileage,horsepower,is_new,actual_price,predicted_price,absolute_error
1478429,2015,Porsche,Rare,Unknown,1034.0,395.0,False,1225800.0,239518.107893,9.862819e+05
2544096,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1116711.0,63713.815045,1.052997e+06
2544074,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1115276.0,63713.815045,1.051562e+06
2544045,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1114461.0,63713.815045,1.050747e+06
488893,2005,Porsche,Rare,2 Dr STD Convertible,338.0,605.0,False,984900.0,203786.967589,7.811130e+05
2256267,2017,Ford,GT,RWD,80.0,647.0,False,975000.0,283835.479851,6.911645e+05
59100,2005,Porsche,Rare,2 Dr STD Convertible,2300.0,605.0,False,819000.0,200191.110366,6.188089e+05
2610762,2006,Land Rover,LR3,V6,120000.0,216.0,False,699900.0,7892.077698,6.920079e+05
1166533,2020,Ford,F-250 Super Duty,Unknown,3.0,445.0,True,589250.0,61889.380443,5.273606e+05
941607,1989,BMW,7 Series,750iL RWD,41308.0,300.0,False,4500.0,586293.889209,5.817939e+05


In [37]:
mercedes_mask = (
    (X_val["year"] == 2021) &
    (X_val["make_name"] == "Mercedes-Benz") &
    (X_val["model_name"] == "E-Class")
)

In [38]:
y_val.loc[mercedes_mask].sort_values(ascending=False)

2544096    1116711.0
2544074    1115276.0
2544045    1114461.0
2536065      59475.0
2535528      57500.0
Name: price, dtype: float64

In [39]:
y_val.loc[ 2536065 ]

np.float64(59475.0)

In [40]:
X_val.loc[[2544096, 2544074, 2544045]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
2544096,Foothill Ranch,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,0.0,E-Class,...,Not Reported,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False
2544074,Foothill Ranch,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,0.0,E-Class,...,Not Reported,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False
2544045,Foothill Ranch,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,0.0,E-Class,...,Not Reported,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False


In [41]:
rows = X_val.loc[[2544096, 2544074, 2544045]]

rows.nunique()

city                        1
engine_type                 1
frame_damaged               1
fuel_type                   1
has_accidents               1
horsepower                  1
is_new                      1
make_name                   1
mileage                     1
model_name                  1
owner_count                 1
salvage                     1
transmission                1
trim_name                   1
wheel_system                1
year                        1
mileage_missing             1
horsepower_missing          1
new_mileage_conflict        1
used_owner_count_missing    1
Condition_reported          1
dtype: int64

In [43]:
reference_car = X_val.loc[2544096]

In [44]:
same_car_mask = X_train.eq(reference_car).all(axis=1)

In [45]:
same_car_mask.sum()

np.int64(1)

In [46]:
X_train.loc[same_car_mask]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
2543332,Foothill Ranch,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,0.0,E-Class,...,Not Reported,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False


In [47]:
y_train.loc[same_car_mask]

2543332    61405.0
Name: price, dtype: float64

In [48]:
worst_cases.sort_values("actual_price", ascending = False)

,year,make_name,model_name,trim_name,mileage,horsepower,is_new,actual_price,predicted_price,absolute_error
1478429,2015,Porsche,Rare,Unknown,1034.0,395.0,False,1225800.0,239518.107893,9.862819e+05
2544096,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1116711.0,63713.815045,1.052997e+06
2544074,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1115276.0,63713.815045,1.051562e+06
2544045,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1114461.0,63713.815045,1.050747e+06
488893,2005,Porsche,Rare,2 Dr STD Convertible,338.0,605.0,False,984900.0,203786.967589,7.811130e+05
2256267,2017,Ford,GT,RWD,80.0,647.0,False,975000.0,283835.479851,6.911645e+05
59100,2005,Porsche,Rare,2 Dr STD Convertible,2300.0,605.0,False,819000.0,200191.110366,6.188089e+05
2610762,2006,Land Rover,LR3,V6,120000.0,216.0,False,699900.0,7892.077698,6.920079e+05
1166533,2020,Ford,F-250 Super Duty,Unknown,3.0,445.0,True,589250.0,61889.380443,5.273606e+05
941607,1989,BMW,7 Series,750iL RWD,41308.0,300.0,False,4500.0,586293.889209,5.817939e+05


In [49]:
Range_Rover_mask = (
    (X_val["year"] == 2006) &
    (X_val["make_name"] == "Land Rover") &
    (X_val["model_name"] == "LR3")
)

In [50]:
y_val.loc[Range_Rover_mask].sort_values(ascending=False)

2610762    699900.0
1006302     10999.0
741744       6999.0
169660       6650.0
Name: price, dtype: float64

In [54]:
X_val.loc[[2610762, 1006302, 741744, 169660]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
2610762,Modesto,V6,True,Gasoline,True,216.0,False,Land Rover,120000.0,LR3,...,False,A,V6,4WD,2006,0,0,False,False,True
1006302,Roswell,V8,False,Gasoline,True,300.0,False,Land Rover,127919.0,LR3,...,False,A,HSE,4WD,2006,0,0,False,False,True
741744,Raleigh,V8,False,Gasoline,False,300.0,False,Land Rover,168523.0,LR3,...,False,A,SE,4WD,2006,0,0,False,False,True
169660,Naugatuck,V8,False,Gasoline,True,300.0,False,Land Rover,157405.0,LR3,...,False,A,SE,4WD,2006,0,0,False,False,True


In [62]:
Porsche_mask_2005 = (
    (X_val["year"] == 2005) &
    (X_val["make_name"] == "Porsche") &
    (X_val["model_name"] == "Rare")
)

In [65]:
Porsche_mask_2015 = (
    (X_val["year"] == 2015) &
    (X_val["make_name"] == "Porsche") &
    (X_val["model_name"] == "Rare")
)

In [63]:
y_val.loc[Porsche_mask_2005].sort_values(ascending=False)

488893    984900.0
59100     819000.0
Name: price, dtype: float64

In [66]:
y_val.loc[Porsche_mask_2015].sort_values(ascending=False)

1478429    1225800.0
Name: price, dtype: float64

In [68]:
X_val.loc[[488893, 59100,1478429]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
488893,Hinsdale,V10,False,Gasoline,True,605.0,False,Porsche,338.0,Rare,...,False,M,2 Dr STD Convertible,RWD,2005,0,0,False,False,True
59100,Parsippany-Troy Hills,V10,False,Gasoline,False,605.0,False,Porsche,2300.0,Rare,...,False,M,2 Dr STD Convertible,RWD,2005,0,0,False,False,True
1478429,West Chicago,V8,False,Gasoline,False,395.0,False,Porsche,1034.0,Rare,...,False,A,Unknown,Unknown,2015,0,1,False,False,True


In [81]:
X_val.loc[[1478429]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1478429,West Chicago,V8,False,Gasoline,False,395.0,False,Porsche,1034.0,Rare,...,False,A,Unknown,Unknown,2015,0,1,False,False,True


In [69]:
Ford_GT_mask = (
    (X_val["year"] == 2017) &
    (X_val["make_name"] == "Ford") &
    (X_val["model_name"] == "GT")
)

In [73]:
Ford_F_250SuperDuty_mask = (
    (X_val["year"] == 2020) &
    (X_val["make_name"] == "Ford") &
    (X_val["model_name"] == "F-250 Super Duty")
)

In [71]:
y_val.loc[Ford_GT_mask].sort_values(ascending=False)

2256267    975000.0
Name: price, dtype: float64

In [74]:
y_val.loc[Ford_F_250SuperDuty_mask].sort_values(ascending=False)

1166533    589250.0
635074     104136.0
777486     100458.0
708421      99995.0
2100566     98750.0
             ...   
2602089     36033.0
2543217     35548.0
2527829     35305.0
381260      35211.0
1923316     29783.0
Name: price, Length: 718, dtype: float64

In [77]:
X_val.loc[[1166533, 635074, 777486, 708421, 2100566,2602089,2543217,2527829, 381260, 1923316]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1166533,Chiefland,V8 Biodiesel,Not Reported,Biodiesel,Not Reported,445.0,True,Ford,3.0,F-250 Super Duty,...,Not Reported,Unknown,Unknown,Unknown,2020,0,1,False,False,False
635074,Antioch,V8 Biodiesel,False,Biodiesel,False,475.0,True,Ford,5.0,F-250 Super Duty,...,False,A,Lariat Crew Cab LB 4WD,4WD,2020,0,0,False,False,True
777486,Greensboro,V8 Biodiesel,False,Biodiesel,False,445.0,True,Ford,54.0,F-250 Super Duty,...,False,A,Unknown,Unknown,2020,0,1,False,False,True
708421,Glen Allen,V8 Biodiesel,False,Biodiesel,False,475.0,True,Ford,20.0,F-250 Super Duty,...,False,A,Lariat Crew Cab LB 4WD,4WD,2020,0,0,False,False,True
2100566,Georgetown,V8 Biodiesel,False,Biodiesel,False,445.0,True,Ford,0.0,F-250 Super Duty,...,False,A,Unknown,Unknown,2020,0,1,False,False,True
2602089,Salinas,V8 Flex Fuel Vehicle,Not Reported,Flex Fuel Vehicle,Not Reported,385.0,True,Ford,186.0,F-250 Super Duty,...,Not Reported,A,XL LB RWD,4X2,2020,0,0,False,False,False
2543217,Riverside,V8 Flex Fuel Vehicle,Not Reported,Flex Fuel Vehicle,Not Reported,385.0,True,Ford,53.0,F-250 Super Duty,...,Not Reported,A,XL LB RWD,4X2,2020,0,0,False,False,False
2527829,Rancho Santa Margarita,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,385.0,True,Ford,15.0,F-250 Super Duty,...,False,A,XL LB RWD,4X2,2020,0,0,False,False,True
381260,Baltimore,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,385.0,True,Ford,3.0,F-250 Super Duty,...,False,A,XL LB 4WD,4WD,2020,0,0,False,False,True
1923316,Morrilton,V8 Flex Fuel Vehicle,Not Reported,Flex Fuel Vehicle,Not Reported,381.0,True,Ford,17.0,F-250 Super Duty,...,Not Reported,A,Unknown,Unknown,2020,0,1,False,False,False


In [85]:
BMW_mask = (
    (X_val["year"] == 1989) &
    (X_val["make_name"] == "BMW") &
    (X_val["model_name"] == "7 Series")
)

In [86]:
y_val.loc[BMW_mask].sort_values(ascending=False)

941607    4500.0
Name: price, dtype: float64

In [87]:
X_val.loc[[ 941607]]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
941607,Greenville,V12,False,Gasoline,False,300.0,False,BMW,41308.0,7 Series,...,False,A,750iL RWD,RWD,1989,1,0,False,False,True


Current stage: Baseline evaluation + error analysis

We have not moved on to a stronger model yet. That would be premature because the current investigation is showing us that part of the error comes from the data itself.

What is already done
Data loaded and cleaned
Train / validation / test split created
Feature engineering completed
Numerical features scaled
Categorical features one-hot encoded
Boolean features passed through
Linear Regression baseline trained
Validation predictions made
Training predictions made
Naive baseline calculated
Current metrics
Naive mean-price baseline RMSE: about $19,631
Linear Regression train RMSE: about $7,671
Linear Regression validation RMSE: about $7,636
Validation MAE: about $3,078
Validation R²: about 0.849
What those metrics tell us
Linear Regression is clearly learning useful information because it beats the dumb baseline heavily.
Train and validation RMSE are almost identical.
So high variance / classic overfitting does not appear to be the main issue.
We cannot confidently call it high bias yet because some of the RMSE is clearly being inflated by bad/noisy target values.
What error analysis has uncovered

We sorted the largest validation errors and inspected the actual cars behind them.

1. Likely corrupted target prices

2021 Mercedes E-Class

Actual labels around $1.11M
Comparable listings around $57k–$61k
Model predicted about $64k
Three feature-identical validation rows
Identical-feature training row priced about $61k

2006 Land Rover LR3

Suspicious row: about $700k
Comparable LR3s: about $6k–$11k

2020 Ford F-250

Suspicious row: about $589k
Most comparable vehicles: roughly $30k–$100k

These look much more like data-quality problems than model failures.

2. Legitimate extreme vehicles

Some huge prices may actually be valid.

Examples:

Ford GT
Some rare Porsche V10 / 605 hp listings

This taught us an important rule:

Extreme price does not automatically mean bad data.

So we cannot just delete everything above some arbitrary price threshold.

3. Genuine model / representation failures

1989 BMW 750iL

Actual around $4,500
Model predicted around $586k

That is much more likely a genuine model failure.

We also uncovered a possible feature-engineering issue:

rare model_name → "Rare"

That may have thrown away valuable identity information for exotic vehicles.

A rare Porsche being reduced to simply "Rare" can make very different exotic cars look too similar to the model.

Another issue we discovered
Repeated listings crossing splits

The same feature-identical Mercedes configuration existed in both:

training
validation

That means our random split may allow duplicate/repeated listings to cross splits.

That could make validation less representative of truly unseen listings.

We have not yet measured how widespread this is.

ERROR ANALYSIS 
1. INVESTIGATE PORSCHE RARE COLUMNS. 
WHAT WAS FOUND 

Carrera GT: 2 train, 2 validation, 0 test
918 Spyder: 3 train, 1 validation, 0 test
Hypothesis:
The large Porsche prediction errors were caused by our rare-category rule. Models appearing fewer than 10 times in training were replaced with "Rare", which may have removed important vehicle identity.

Investigation:
The original data showed that the affected cars were Porsche Carrera GTs and 918 Spyders, with multiple listings in similarly high price ranges. After feature engineering, those model names became "Rare" in X_train_10 / X_val_10.

Conclusion:
The prices appear legitimate. The problem is likely information loss from grouping rare model_name values. For exotic cars, rarity is closely tied to value, so collapsing them into "Rare" can cause severe underprediction.

2. INVESTIGATE BMW 
WHAT WAS FOUND

A corrupted $1.75M BMW 750iL training label likely distorted what Linear Regression learned for that rare trim, contributing to the absurd ~$586k prediction on the 1989 BMW.
3. WRITE A TABLE TO CLASSIFY GROUPS ie

corrupted targets
legitimate expensilve
repeated listing across splits 
Rare grouping info loss
genuine model failures

In [88]:
worst_cases.sort_values("actual_price", ascending = False)

,year,make_name,model_name,trim_name,mileage,horsepower,is_new,actual_price,predicted_price,absolute_error
1478429,2015,Porsche,Rare,Unknown,1034.0,395.0,False,1225800.0,239518.107893,9.862819e+05
2544096,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1116711.0,63713.815045,1.052997e+06
2544074,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1115276.0,63713.815045,1.051562e+06
2544045,2021,Mercedes-Benz,E-Class,E 350 Sedan RWD,0.0,180.0,True,1114461.0,63713.815045,1.050747e+06
488893,2005,Porsche,Rare,2 Dr STD Convertible,338.0,605.0,False,984900.0,203786.967589,7.811130e+05
2256267,2017,Ford,GT,RWD,80.0,647.0,False,975000.0,283835.479851,6.911645e+05
59100,2005,Porsche,Rare,2 Dr STD Convertible,2300.0,605.0,False,819000.0,200191.110366,6.188089e+05
2610762,2006,Land Rover,LR3,V6,120000.0,216.0,False,699900.0,7892.077698,6.920079e+05
1166533,2020,Ford,F-250 Super Duty,Unknown,3.0,445.0,True,589250.0,61889.380443,5.273606e+05
941607,1989,BMW,7 Series,750iL RWD,41308.0,300.0,False,4500.0,586293.889209,5.817939e+05
